In [13]:
import pandas as pd
import numpy as np

In [14]:
import time
import mlflow
import mlflow.sklearn

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

TRACKING_URI = "http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("booking-price-predictor")

<Experiment: artifact_location='s3://yt-plugin-mlflow/3', creation_time=1780389781903, experiment_id='3', last_update_time=1780389781903, lifecycle_stage='active', name='booking-price-predictor', tags={}, trace_location=None, workspace='default'>

In [15]:
def log_model_run(
    model,
    model_name,
    X_train,
    y_train,
    X_test,
    y_test,
    params=None
):

    with mlflow.start_run(run_name=model_name):

        mlflow.set_tags({
            "project": "RideML",
            "stage": "model_selection",
            "dataset": "nyc_taxi_fare"
        })

        start_time = time.time()

        model.fit(X_train, y_train)

        training_time = time.time() - start_time

        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = mean_squared_error(y_test, y_pred) ** 0.5
        r2 = r2_score(y_test, y_pred)

        # Params
        mlflow.log_param("algorithm", model_name)

        if params:
            mlflow.log_params(params)

        # Metrics
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("training_time_sec", training_time)

        # Dataset metadata
        mlflow.log_param("train_rows", len(X_train))
        mlflow.log_param("test_rows", len(X_test))
        mlflow.log_param("features", X_train.shape[1])

        # Save model
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model"
        )

        print("=" * 50)
        print(model_name)
        print(f"MAE  : {mae:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"R²   : {r2:.4f}")
        print("=" * 50)

In [16]:
import pandas as pd

df_train = pd.read_csv(r"C:\Users\kadam\Desktop\Ride-Booking-Value-Prediction\data\processed\train.csv")

df_test = pd.read_csv(r"C:\Users\kadam\Desktop\Ride-Booking-Value-Prediction\data\processed\test.csv")

# Split features and target
X_train = df_train.drop('fare_amount', axis=1)
y_train = df_train['fare_amount']

X_test = df_test.drop('fare_amount', axis=1)
y_test = df_test['fare_amount']

In [17]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

log_model_run(
    model=lr,
    model_name="LinearRegression",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

2026/06/02 14:53:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 14:53:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LinearRegression
MAE  : 2.3658
RMSE : 5.1112
R²   : 0.7212
🏃 View run LinearRegression at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/0c0b8fa7f7b34c4a95960dc56146fb41
🧪 View experiment at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly = Pipeline([
    (
        "poly",
        PolynomialFeatures(
            degree=3,
            include_bias=False
        )
    ),
    (
        "lr",   
        LinearRegression()
    )
])

log_model_run(
    model=poly,
    model_name="PolynomialRegression_Degree3",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "degree": 3
    }
)

2026/06/02 14:54:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 14:54:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


PolynomialRegression_Degree3
MAE  : 2.1120
RMSE : 4.3824
R²   : 0.7950
🏃 View run PolynomialRegression_Degree3 at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/8f1572d114444680b2c2cbeb6aa5be29
🧪 View experiment at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3


In [19]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

log_model_run(
    model=gbr,
    model_name="GradientBoosting",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "n_estimators": 300,
        "learning_rate": 0.05
    }
)

2026/06/02 14:56:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 14:56:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


GradientBoosting
MAE  : 1.9409
RMSE : 3.9370
R²   : 0.8346
🏃 View run GradientBoosting at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/9e36bc343d8e4119bad7a2df5f0e9fd6
🧪 View experiment at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3


In [20]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

log_model_run(
    model=rf,
    model_name="RandomForest",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "n_estimators": 300
    }
)

2026/06/02 14:59:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 14:59:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest
MAE  : 1.8303
RMSE : 3.8510
R²   : 0.8417
🏃 View run RandomForest at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/ebb45a7d1acd42f6b26eb4f7d057b3b0
🧪 View experiment at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3


In [21]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

log_model_run(
    model=xgb,
    model_name="XGBoost",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 8
    }
)

2026/06/02 15:04:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 15:04:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost
MAE  : 1.6711
RMSE : 3.7336
R²   : 0.8512
🏃 View run XGBoost at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/62dabb4aa5b0434099278fe9495ade11
🧪 View experiment at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3


In [22]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

log_model_run(
    model=lgbm,
    model_name="LightGBM",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 8
    }
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001790 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1363
[LightGBM] [Info] Number of data points in the train set: 156629, number of used features: 12
[LightGBM] [Info] Start training from score 11.345584


2026/06/02 15:04:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 15:04:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LightGBM
MAE  : 1.7494
RMSE : 3.7096
R²   : 0.8531
🏃 View run LightGBM at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3/runs/e7cc0ff1ca5140bb9e051fc4023ea46d
🧪 View experiment at: http://ec2-16-171-45-136.eu-north-1.compute.amazonaws.com:5000/#/experiments/3


In [23]:
results = []

for name, model in {
    "RandomForest": rf,
    "XGBoost": xgb,
    "LightGBM": lgbm,
    "GrafientBoosting": gbr,
    "PolynomialRegression_Degree3": poly
}.items():

    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values("R2", ascending=False))

                          Model       MAE      RMSE        R2
2                      LightGBM  1.749407  3.709558  0.853144
1                       XGBoost  1.671079  3.733596  0.851235
0                  RandomForest  1.830294  3.850979  0.841733
3              GrafientBoosting  1.940894  3.937029  0.834581
4  PolynomialRegression_Degree3  2.112008  4.382374  0.795041
